# 🏋️ 코치 목소리 인공지능 학습기
방금 만드신 데이터셋(`voice_dataset_final.zip`)을 이용해 AI에게 코치님의 목소리를 가르치는(학습) 단계입니다.

## 1단계: 압축 파일 업로드
먼저 방금 컴퓨터에 다운로드 받았던 `voice_dataset_final.zip` 파일을 왼쪽 폴더 아이콘(📁) 영역에 드래그 앤 드롭해서 업로드해 주세요.

## 2단계: 필수 프로그램 설치 (▶️ 클릭)
목소리 학습을 위한 프레임워크(Piper)를 설치합니다. (약 1~2분 소요)

In [ ]:
!rm -rf /content/piper-train
!git clone https://github.com/rhasspy/piper.git
!mv piper/src/python /content/piper-train
!rm -rf piper
%cd /content/piper-train
!python3 -c "import sys; [open(fn, 'w').write(open(fn, 'r').read().replace('piper-phonemize~=1.1.0', 'piper-phonemize-fix')) for fn in ['requirements.txt', 'setup.py']]"
!pip install -q -r requirements.txt
!python3 -m pip install -q -e .

# 학습 핵심 모듈 컴파일
    # Python 3.12 환경을 위해 setup.py를 최신 문법(setuptools)으로 완전히 재작성
    setup_code = """\
from setuptools import setup, Extension
from Cython.Build import cythonize
import numpy
ext = Extension(name="core", sources=["core.pyx"], include_dirs=[numpy.get_include()])
setup(name="monotonic_align", ext_modules=cythonize(ext))
"""
    with open('piper_train/vits/monotonic_align/setup.py', 'w') as f:
        f.write(setup_code)

%cd piper_train/vits/monotonic_align
!python3 setup.py build_ext --inplace
    # monotonic_align 패키지 참조 버그 수정
    init_file = 'piper_train/vits/monotonic_align/__init__.py'
    with open(init_file, 'r') as f:
        content = f.read()
    if 'from .monotonic_align.core import' in content:
        content = content.replace('from .monotonic_align.core import', 'from .core import')
        with open(init_file, 'w') as f:
            f.write(content)
%cd /content/piper-train

print("\n✅ 필수 프로그램 설치 완료!")

## 3단계: 파일 압축 해제 및 기본 뼈대 준비 (▶️ 클릭)
업로드한 파일의 압축을 풀고, 맨바닥에서 시작하면 너무 오래 걸리니 '기본 한국인 목소리' 모델을 가져와서 코치님 목소리로 덧씌우기(Fine-tuning)를 준비합니다.

In [ ]:
import os
if not os.path.exists('/content/voice_dataset_final.zip'):
    print('❌ 에러: voice_dataset_final.zip 파일이 없습니다! 왼쪽 폴더에 업로드가 끝날 때까지 기다려 주세요.')
else:
    !unzip -q -o /content/voice_dataset_final.zip -d /content/dataset
    print('✅ 파일 압축 해제 완료!')
    
    print('기본 한국어 모델 다운로드 중...')
    !wget -q -O /content/ko_base.ckpt https://huggingface.co/rhasspy/piper-voices/resolve/main/ko/ko_KR/kss/medium/ko_KR-kss-medium.ckpt
    print('✅ 준비 완료!')

## 4단계: 데이터 전처리 (▶️ 클릭)
AI가 텍스트를 읽을 수 있는 발음 기호로 변환하는 등 데이터를 씹기 좋게 다듬습니다.

In [ ]:
!python3 -m piper_train.preprocess \
  --language ko \
  --input-dir /content/dataset \
  --output-dir /content/training \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050
  
print("\n✅ 전처리 완료! 이제 본격적인 학습이 가능합니다.")

## 5단계: 코치 목소리 학습 시작! (▶️ 클릭)
실제 딥러닝 학습을 시작합니다. **이 과정은 데이터 양에 따라 수십 분에서 몇 시간**이 걸릴 수 있습니다.
중간에 에포크(Epoch)가 올라가는 것을 볼 수 있습니다. 충분히 학습된 것 같으면 정지(■) 버튼을 눌러도 됩니다.

In [ ]:
!python3 -m pytorch_lightning.utilities.upgrade_checkpoint /content/ko_base.ckpt

# 최대 1000회 반복(Epoch) 학습합니다. 원할 때 중지하세요.
!python3 -m piper_train \
  --dataset-dir /content/training \
  --accelerator 'gpu' \
  --devices 1 \
  --batch-size 8 \
  --validation-split 0.0 \
  --num-test-examples 0 \
  --max_epochs 1000 \
  --resume_from_checkpoint /content/ko_base.ckpt \
  --checkpoint-epochs 50 \
  --precision 32

## 6단계: 학습된 모델 웹용으로 추출 및 다운로드 (▶️ 클릭)
학습을 끝내고(정지한 후) 이 버튼을 누르면 브라우저용 `coach_voice.onnx` 파일이 만들어지고 자동 다운로드됩니다.

In [ ]:
import os
import glob
import subprocess

print('🔄 Google Colab의 최신 업데이트(PyTorch 2.6)로 인한 호환성 문제를 해결하기 위해 안정 버전으로 다운그레이드 합니다...')
print('⏳ 약 1~2분 정도 소요될 수 있습니다. 잠시만 기다려주세요!')
os.system('pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121')

print('🧹 충돌을 일으킬 수 있는 onnxscript 라이브러리를 안전하게 제거합니다...')
os.system('pip uninstall -y onnxscript')

# 1. 작업 디렉터리를 환경에 맞게 변경
if os.path.exists('/content/piper-train'):
    work_dir = '/content/piper-train'
elif os.path.exists('/content/piper/src/python'):
    work_dir = '/content/piper/src/python'
else:
    work_dir = None
    print('❌ 에러: 파이퍼(piper) 폴더를 찾을 수 없습니다.')

if work_dir:
    os.chdir(work_dir)
    print(f'📁 작업 디렉터리 이동 완료: {os.getcwd()}')
    
    # 원상 복구 (이전에 추가했던 fallback=True 패치 제거)
    export_script = os.path.join(work_dir, 'piper_train', 'export_onnx.py')
    if os.path.exists(export_script):
        with open(export_script, 'r') as f:
            code = f.read()
        if 'fallback=True, ' in code:
            code = code.replace('fallback=True, ', '')
            with open(export_script, 'w') as f:
                f.write(code)
    
    # 2. 누락되었던 핵심 C++ 모듈 강제 컴파일
    align_dir = os.path.join(work_dir, 'piper_train', 'vits', 'monotonic_align')
    if os.path.exists(align_dir):
        os.chdir(align_dir)
        os.system('python3 setup.py build_ext --inplace')
    # monotonic_align 패키지 참조 버그 수정
    init_file = 'piper_train/vits/monotonic_align/__init__.py'
    with open(init_file, 'r') as f:
        content = f.read()
    if 'from .monotonic_align.core import' in content:
        content = content.replace('from .monotonic_align.core import', 'from .core import')
        with open(init_file, 'w') as f:
            f.write(content)
        os.chdir(work_dir)
    
    # 3. 가장 최근에 학습된 체크포인트 파일(*.ckpt) 검색
    checkpoints = glob.glob('/content/training/lightning_logs/version_*/checkpoints/*.ckpt')
    
    if not checkpoints:
        print('❌ 에러: 학습된 체크포인트(.ckpt) 파일을 찾을 수 없습니다.')
    else:
        latest_ckpt = max(checkpoints, key=os.path.getctime)
        print(f'🎯 가장 최근에 학습된 체크포인트 선택: {latest_ckpt}')
        
        # 4. ONNX 모델 추출 명령 실행
        print('📦 ONNX 모델 추출 중... (버전 다운그레이드 및 충돌 제거 완료)')
        cmd = f'python3 -m piper_train.export_onnx {latest_ckpt} /content/coach_voice.onnx'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        
        if result.returncode == 0:
            print('✅ ONNX 모델 추출 성공!')
            from google.colab import files
            print('⬇️ 완성된 목소리 모델을 컴퓨터로 다운로드합니다!')
            files.download('/content/coach_voice.onnx')
        else:
            print(f'❌ 에러: ONNX 모델 추출에 실패했습니다. (코드: {result.returncode})')
    # Python 3.12 환경을 위해 setup.py를 최신 문법(setuptools)으로 완전히 재작성
    setup_code = """\
from setuptools import setup, Extension
from Cython.Build import cythonize
import numpy
ext = Extension(name="core", sources=["core.pyx"], include_dirs=[numpy.get_include()])
setup(name="monotonic_align", ext_modules=cythonize(ext))
"""
    with open('piper_train/vits/monotonic_align/setup.py', 'w') as f:
        f.write(setup_code)

            print(result.stderr[-2000:])
